# 04 · Retrain and export the models

This notebook reproduces the two bundled emotion models from scratch and shows how to export
and use your own:

1. train PLS(31) and kernel ridge with the person-wise 70/20/10 split;
2. compare them with the bundled models;
3. inspect what is stored with a model (metadata, calibration target, reliability reference);
4. export to a portable `.npz` file that needs only numpy, and load it back;
5. use the model in Python, on the command line and in the demos.

The same steps are available as `fea train` (see the README). Runtime: about 1 minute.

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="IProgress not found")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 3)
ROOT = Path.cwd() if (Path.cwd() / "examples").is_dir() else Path.cwd().parent  # repo root

import time

from facial_expression_analysis import (
    DlibLandmarkDetector,
    EmotionEstimator,
    KernelEmotionModel,
    LinearEmotionModel,
    load_image,
    load_model,
)
from facial_expression_analysis.training import Estimator, TrainConfig, train_model
from facial_expression_analysis.training.data import load_morphset

MODELS = ROOT / "models"  # not tracked by git

## 1. Train

`TrainConfig` holds every choice made in notebooks 01–03: the 51 non-jaw landmarks, the
stable scale, and the estimator and its hyper-parameters. The split is by person, with seed 1
as for the bundled models.

In [2]:
data = load_morphset()
configs = {
    "PLS(31)": TrainConfig(),
    "kernel ridge": TrainConfig(estimator=Estimator.KERNEL_RIDGE),
}
for name, cfg in configs.items():
    print(f"{name:13s} {cfg.describe()}")

PLS(31)       PLSRegression(n_components=31)
kernel ridge  StandardScaler + Nystroem(rbf, gamma=0.5/n_features, n_components=2000) + Ridge(alpha=1.0)


In [3]:
%%time
results = {}
for name, cfg in configs.items():
    t0 = time.perf_counter()
    results[name] = train_model(data, cfg)
    print(
        f"{name}: trained on {results[name].model.metadata['n_train']:,} faces in {time.perf_counter() - t0:.0f} s"
    )

PLS(31): trained on 116,352 faces in 30 s


kernel ridge: trained on 116,352 faces in 13 s
CPU times: user 42.2 s, sys: 647 ms, total: 42.9 s
Wall time: 42.7 s


## 2. Compare with the bundled models

Test CCC on the 10% of persons held out from training and validation. The pipeline view clips
outputs to their valid ranges, as users see them.

In [4]:
bundled = {"PLS(31)": LinearEmotionModel.load(), "kernel ridge": KernelEmotionModel.load()}
rows = []
for name in configs:
    for source, metrics in [
        ("retrained", results[name].metrics),
        ("bundled", bundled[name].metadata["metrics"]),
    ]:
        test = metrics["test"]["pipeline"]
        rows.append(
            {
                "model": name,
                "source": source,
                **{t: test[t]["ccc"] for t in ("arousal", "valence", "intensity")},
            }
        )
pd.DataFrame(rows).set_index(["model", "source"]).round(4)

arousal  valence  intensity
model        source                                
PLS(31)      retrained    0.606    0.791      0.645
             bundled      0.605    0.791      0.645
kernel ridge retrained    0.703    0.831      0.722
             bundled      0.703    0.831      0.722

The retrained models match the bundled ones (identical split and settings; tiny differences
can come from library versions).

## 3. What a model file contains

Besides the weights, each model stores how it was trained and two references used at run time:

- `neutral_mean`: the average raw prediction for neutral faces of validation persons. It is
  the target of per-person calibration in the demo.
- `basis_distance_quantiles` (kernel model): the distance from validation faces to their nearest
  training faces. Above the 99th percentile, the landmarks have probably fitted badly; the demos
  show this as a reliability badge.

In [5]:
pls, kernel = results["PLS(31)"].model, results["kernel ridge"].model
meta = {k: v for k, v in pls.metadata.items() if k not in ("metrics", "config")}
meta

{'estimator': 'PLSRegression(n_components=31)',
 'landmark_subset': 'no_jaw',
 'scale_reference': 'stable',
 'split': 'by person (see facial_expression_analysis.training.data)',
 'trained_at': '2026-11-15T02:08:58+00:00',
 'package_version': '2.0.0',
 'sklearn_version': '1.9.1',
 'python_version': '3.12.16',
 'n_train': 116352,
 'neutral_mean': [0.09489543883650352,
  -0.11838338641657033,
  0.3163572300798835]}

In [6]:
print("kernel model neutral_mean:", np.round(kernel.metadata["neutral_mean"], 3))
print(
    "basis distance quantiles:",
    {q: round(v, 1) for q, v in kernel.metadata["basis_distance_quantiles"].items()},
)
print(
    f"PLS: affine map {pls.coef.shape} + intercept; kernel: basis of {len(kernel.basis_landmarks)} faces"
)

kernel model neutral_mean: [ 0.062 -0.169  0.291]
basis distance quantiles: {'0.5': 16.2, '0.95': 22.9, '0.99': 26.3}
PLS: affine map (1275, 3) + intercept; kernel: basis of 2000 faces


## 4. Export and reload

PLS is collapsed into one affine map, and the kernel model stores its Nyström basis as frontal
landmarks. Both are saved as compressed `.npz` files with JSON metadata. Loading them needs
only numpy, not scikit-learn.

In [7]:
paths = {
    "PLS(31)": pls.save(MODELS / "notebook_pls31.npz"),
    "kernel ridge": kernel.save(MODELS / "notebook_kernel.npz"),
}
X = pls.features.transform(EmotionEstimator().frontalizer(data.landmarks[:2000]))
for name, path in paths.items():
    reloaded = load_model(path)
    same = np.allclose(
        reloaded.predict_features(X), results[name].model.predict_features(X), atol=1e-6
    )
    t0 = time.perf_counter()
    reloaded.predict_features(X)
    us = (time.perf_counter() - t0) / len(X) * 1e6
    print(
        f"{name:13s} {path.relative_to(ROOT)}  {path.stat().st_size / 1024:,.0f} KB  "
        f"reload identical: {same}  {us:.1f} µs per face"
    )

PLS(31)       models/notebook_pls31.npz  31 KB  reload identical: True  0.6 µs per face


kernel ridge  models/notebook_kernel.npz  2,100 KB  reload identical: True  22.8 µs per face


## 5. Use it

In Python, pass the model to the estimator:

In [8]:
image = load_image(ROOT / "examples" / "happy.jpg")
detector = DlibLandmarkDetector()
for name, path in paths.items():
    (face,) = EmotionEstimator(load_model(path)).analyze_image(image, detector)
    e = face.emotion
    print(f"{name:13s} {e.label}  (A {e.arousal:+.2f}, V {e.valence:+.2f}, I {e.intensity:.2f})")

PLS(31)       Very delighted  (A +0.35, V +0.44, I 0.77)
kernel ridge  Extremely delighted  (A +0.41, V +0.55, I 0.83)


On the command line and in the OpenCV demo:

```bash
uv run fea predict examples/happy.jpg --model models/notebook_pls31.npz
uv run fea evaluate --model models/notebook_pls31.npz
uv run fea webcam --model models/notebook_pls31.npz
```

To make a model the default everywhere, including the web demo and the Hugging Face Space,
copy it over the bundled file and run the tests:

```bash
cp models/notebook_pls31.npz src/facial_expression_analysis/assets/emotion_model.npz
uv run pytest
```

**Next:** notebook 05 analyses where the model errs and what it has learned.